# UTS Machine Learning 2026 – Task 1: Estimasi Waktu Pengiriman (`durasi_jam`)

**Platform:** Python di VS Code (`.venv`) &nbsp;|&nbsp; **Modul Utama:** `kode/regresi.py` & `kode/config.py`

**Tujuan:** Memprediksi median (`p50`) dan batas atas 90% (`p90`) waktu pengiriman (`durasi_jam`) untuk paket uji **September–Oktober 2025** dengan **hanya menggunakan fitur yang tersedia saat paket diambil kurir (pickup)**, tanpa kebocoran data masa depan.

| Bagian | Cakupan Ketentuan UTS | Deskripsi |
|---|---|---|
| 0 | R3 | Persiapan lingkungan, self-healing `sys.path`, versi library eksplisit, dan `SEED = 42` |
| 1 | R1a | Load data latih (Jan–Agu 2025) & uji (Sep–Okt 2025) + ringkasan kamus data |
| 2 | R1a | Audit kebocoran (`jumlah_scan`), kesalahan satuan 60x (`durasi_jam > 1000`), sentinel `jarak_km`, MNAR `berat_kg`, drift `f30`/`f35`, & kategori tak terlihat (`K13`) |
| 3 | R1b | Validasi berbasis waktu (*expanding-window next-month Time CV*) vs Random K-Fold CV |
| 4 | R1c | Feature engineering saat pickup, OOF `TargetEncoder`, baseline + 3 model kuantil lanjutan (*HGBR, LightGBM, XGBoost*) |
| 5 | R1d | Evaluasi Pinball Loss (`q=0.5`, `q=0.9`), cakupan empiris P90, dan kalibrasi Split Conformal (CQR) |
| 6 | R1e | Rincian error per `layanan`, `pulau_tujuan`, `mitra_kurir` (termasuk `K13`), serta interpretabilitas SHAP & PDP |
| 7 | R1f | Ekspor & verifikasi `prediksi_regresi.csv` (`id_paket,p50,p90`, `p90 >= p50`) |
| 8 | R3 | Ringkasan metodologi, keterbatasan, dan reproduksi satu perintah (`python kode/run_all.py`) |

## 0. Persiapan, Versi Library, & Konfigurasi

**Tujuan:** Menyiapkan `PROJECT_ROOT` secara otomatis, mengimpor konfigurasi terpusat (`kode.config`) dan pipeline modular (`kode.regresi`), menetapkan random seed `SEED = 42`, serta mencatat seluruh versi library classical ML yang digunakan.

In [ ]:
from pathlib import Path
import sys

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "kode" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Direktori root proyek yang berisi kode/config.py tidak ditemukan.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from IPython.display import display
from sklearn.inspection import PartialDependenceDisplay

from kode.config import (
    DEFAULT_REGRESI_CSV,
    OUTPUT_DIR,
    SEED,
    format_library_versions,
    get_data_dir,
    get_library_versions,
    get_smoke_config,
    set_global_seed,
)
from kode.regresi import (
    _predict_with_bundle,
    apply_conformal_calibration,
    audit_and_clean_regresi,
    calibrate_conformal_intervals,
    compute_group_breakdowns,
    compute_interpretability,
    evaluate_time_vs_random_cv,
    export_regresi_csv,
    load_regresi_datasets,
    train_and_tune_quantile_models,
)

# ====== KONFIGURASI EKSEKUSI (ubah SMOKE_TEST=True untuk uji cepat) ======
SMOKE_TEST = False
SAMPLE_N = None
DATA_DIR = get_data_dir(must_exist=True)
ARTIFACTS_DIR = OUTPUT_DIR / "regresi_artifacts"
OUTPUT_CSV = DEFAULT_REGRESI_CSV
# ========================================================================

set_global_seed(SEED)
sns.set_theme(style="whitegrid")
lib_versions = get_library_versions()
smoke_cfg = get_smoke_config(smoke_test=SMOKE_TEST, sample_n=SAMPLE_N, seed=SEED)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"OUTPUT_CSV   : {OUTPUT_CSV}")
print(f"SEED         : {SEED}")
print(f"SMOKE_TEST   : {SMOKE_TEST} (sample_n={SAMPLE_N})")
print(f"Versi Library: {format_library_versions(lib_versions)}")

## 1. Memuat Dataset Regresi (`regresi_train.csv.gz` & `regresi_test.csv.gz`)

**Tujuan:** Membaca data latih dan uji dengan presisi `float64` (tanpa downcasting prematur) dan memeriksa rentang tanggal `tgl_pickup` serta kelengkapan kolom.

In [ ]:
train_raw, test_raw = load_regresi_datasets(DATA_DIR, smoke_cfg=smoke_cfg)

print(f"Train shape: {train_raw.shape[0]:,} baris x {train_raw.shape[1]} kolom")
print(f"Test  shape: {test_raw.shape[0]:,} baris x {test_raw.shape[1]} kolom")
print(f"Rentang tgl_pickup train: {train_raw['tgl_pickup'].min()} s.d. {train_raw['tgl_pickup'].max()}")
print(f"Rentang tgl_pickup test : {test_raw['tgl_pickup'].min()} s.d. {test_raw['tgl_pickup'].max()}")
display(train_raw.head(3))

## 2. Audit Kualitas Data & Kebocoran Fitur (R1a)

**Tujuan:** Mengidentifikasi dan mengukur dampak masalah kualitas data serta fitur pasca-pickup:
1. **Kebocoran pasca-pickup (`jumlah_scan`)**: tercatat setelah proses pengiriman berlangsung (`train` berkorelasi kuat dengan `durasi_jam`, sedangkan `test` bernilai kecil `<= 5`). Jika dipakai, prediksi uji akan runtuh secara artifisial. Fitur ini **wajib dibuang** dari seluruh fitur pemodelan.
2. **Kesalahan satuan 60x pada target (`durasi_jam > 1000`)**: nilai ekstrem ribuan jam seluruhnya merupakan menit/satuan 60x; dibuktikan melalui uji kelipatan 60 (`(y * 10) % 60 == 0`), celah distribusi `(700, 1000]`, serta korelasi terhadap `jarak_km` sebelum vs sesudah perbaikan `/ 60.0` (khusus data latih).
3. **Sentinel `jarak_km` (`0.0` & `9999.0`)**: dipulihkan secara hierarkis hanya dari median rute valid di fold latih (`(kota_asal, kota_tujuan) -> (hub_asal, kota_tujuan) -> (kota_asal, pulau_tujuan) -> pulau_tujuan -> global`) + indikator `jarak_km_is_anomalous`.
4. **Missingness MNAR `berat_kg` pada `kargo`**: diimputasi berbasis median `(layanan, volume_bin)` khusus fold latih + indikator `berat_kg_is_missing`.
5. **Pergeseran distribusi (`f30`, `f35`, `antrean_hub_asal`) & kategori uji baru (`mitra_kurir = 'K13'`, `kota_tujuan` 405 level)**.

In [ ]:
train_clean, test_clean, audit_summary = audit_and_clean_regresi(
    train_raw, test_raw, seed=SEED, smoke_cfg=smoke_cfg
)

print("Kolom kebocoran yang dibuang:", audit_summary["dropped_leakage_columns"])
print("\n--- Bukti Kebocoran jumlah_scan ---")
display(pd.DataFrame([audit_summary["jumlah_scan"]]).drop(columns=["impact"], errors="ignore"))
if audit_summary["jumlah_scan"].get("impact"):
    print("Dampak jumlah_scan pada validasi OOT:")
    display(pd.DataFrame([audit_summary["jumlah_scan"]["impact"]]))

print("\n--- Bukti Kesalahan Satuan durasi_jam > 1000 ---")
display(pd.DataFrame([audit_summary["durasi_jam_unit_errors"]]).drop(columns=["impact"], errors="ignore"))
if audit_summary["durasi_jam_unit_errors"].get("impact"):
    print("Dampak pembersihan durasi_jam > 1000 pada validasi OOT:")
    display(pd.DataFrame([audit_summary["durasi_jam_unit_errors"]["impact"]]))

print("\n--- Sentinel jarak_km, MNAR berat_kg, Drift & Profil Kategori ---")
display(pd.DataFrame([audit_summary["jarak_km_sentinels"]]))
display(pd.DataFrame(audit_summary["temporal_drift"]).T)
print("Profil kategori:", audit_summary["categorical_profile"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.log1p(train_raw["durasi_jam"]), bins=50, alpha=0.6, label="Raw durasi_jam")
axes[0].hist(np.log1p(train_clean["durasi_jam"]), bins=50, alpha=0.6, label="Cleaned durasi_jam")
axes[0].set_title("Distribusi log1p(durasi_jam): Sebelum vs Sesudah Perbaikan 60x")
axes[0].set_xlabel("log1p(durasi_jam)")
axes[0].legend()

axes[1].hist(train_raw["jumlah_scan"], bins=30, alpha=0.6, density=True, label="Train jumlah_scan")
axes[1].hist(test_raw["jumlah_scan"], bins=15, alpha=0.7, density=True, label="Test jumlah_scan")
axes[1].set_title("Shift Kebocoran jumlah_scan (Train vs Test)")
axes[1].set_xlabel("jumlah_scan (dibuang dari fitur model)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Validasi Berbasis Waktu vs Random Cross-Validation (R1b)

**Tujuan:** Merancang skema validasi *expanding-window next-month Time-Based CV* (melatih dari bulan awal hingga bulan `t-1` untuk memprediksi bulan `t`, sehingga `train_max < val_min`) dan membandingkannya secara langsung dengan *Random K-Fold CV* pada fitur dan model yang identik untuk mengukur bias optimisme (*optimism gap*).

In [ ]:
cv_summary = evaluate_time_vs_random_cv(train_clean, seed=SEED, smoke_cfg=smoke_cfg)

print("Jendela Fold Time-Based CV (Expanding Window):")
display(pd.DataFrame(cv_summary["time_cv"]["folds"]))

df_time = pd.DataFrame(cv_summary["time_cv"]["models"]).assign(scheme="Time-Based CV")
df_rand = pd.DataFrame(cv_summary["random_cv"]["models"]).assign(scheme="Random K-Fold CV")
df_cv_comp = pd.concat([df_time, df_rand], ignore_index=True)
display(df_cv_comp)

print("Optimism Gap (Time CV - Random CV):")
display(pd.DataFrame(cv_summary["optimism_gap"]))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=df_cv_comp, x="model", y="pinball_avg", hue="scheme", ax=ax)
ax.set_title("Perbandingan Pinball Loss Rata-Rata: Time-Based CV vs Random K-Fold CV")
plt.tight_layout()
plt.show()

## 4. Feature Engineering, Kategori Berkardinalitas Tinggi, & Pemodelan Kuantil (R1c)

**Tujuan:**
- Seluruh transformasi stateful (`RouteJarakImputer`, `BeratMNARImputer`, frekuensi rute, dan OOF `TargetEncoder` pada `kota_tujuan`, `kota_asal`, `mitra_kurir`) dipasang **ketat di dalam setiap fold latih** (*zero leakage*).
- `kota_tujuan` (405 level) dipertahankan sebagai fitur numerik `_te` + `_freq` agar tidak melanggar batas `max_bins=255` pada `HistGradientBoostingRegressor`.
- Kategori baru pada data uji (`mitra_kurir = 'K13'`, atau `kota_tujuan` baru) ditangani aman oleh fallback global prior (`TargetEncoder`), frekuensi `0.0`, `OrdinalEncoder(unknown_value=-1)`, dan masking `pd.Categorical`.
- Membandingkan 2 baseline (`DummyRegressor`, `RidgeLogQuantile`) dan 3 model lanjutan (`HistGradientBoostingRegressor`, `LGBMRegressor`, `XGBRegressor`) dengan tuning hanya di dalam Time-Based CV pada skala `log1p(durasi_jam)`.

In [ ]:
model_results, best_model_bundle = train_and_tune_quantile_models(
    train_clean, seed=SEED, smoke_cfg=smoke_cfg
)

print("Strategi Encoding:", model_results["encoding_strategy"])
print("Model Terpilih   :", model_results["selected_model"])
print("Hyperparameter   :", model_results["selected_params"])

comp_df = pd.DataFrame(model_results["comparison_table"])
display(comp_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
comp_melt = comp_df.melt(
    id_vars=["model"],
    value_vars=["pinball_p50", "pinball_p90"],
    var_name="kuantil",
    value_name="pinball_loss",
)
sns.barplot(data=comp_melt, x="model", y="pinball_loss", hue="kuantil", ax=axes[0])
axes[0].set_title("Perbandingan Pinball Loss (Time-Based CV)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=comp_df, x="model", y="mae", ax=axes[1], color="#4dabf7")
axes[1].set_title("MAE Prediksi Median P50 (jam)")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Ketidakpastian Kuantil (`p50`, `p90`) & Kalibrasi Conformal (R1d)

**Tujuan:** Mengukur Pinball Loss (`q=0.5` dan `q=0.9`), cakupan empiris P90 (`mean(y <= p90)`), serta melakukan kalibrasi *Split Conformal* (membandingkan mode aditif vs *log-space CQR* pada paruh kronologis out-of-time) untuk memastikan cakupan empiris mendekati target `0.90` dan menjamin ketidaksetaraan `p90 >= p50`.

In [ ]:
oot_val_idx = best_model_bundle["oot_val_idx"]
q_hat, oot_p90_cal, calib_summary = calibrate_conformal_intervals(
    y_cal=best_model_bundle["oot_y"],
    p50_cal=best_model_bundle["oot_p50_raw"],
    p90_cal_raw=best_model_bundle["oot_p90_raw"],
    tgl_cal=train_clean.iloc[oot_val_idx]["tgl_pickup"].to_numpy(),
    alpha=0.90,
)

print(f"Ruang Kalibrasi Terpilih : {calib_summary['selected_space']}")
print(f"Conformal q_hat Produksi : {q_hat:.5f}")
display(pd.DataFrame({
    "sebelum_kalibrasi": calib_summary["raw_metrics"],
    "sesudah_kalibrasi": calib_summary["calibrated_metrics"],
}).T)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
cov_before = calib_summary["raw_metrics"]["p90_coverage"]
cov_after = calib_summary["calibrated_metrics"]["p90_coverage"]
ax.bar(["Sebelum Kalibrasi", "Sesudah Conformal"], [cov_before, cov_after], color=["#5c7cfa", "#20c997"])
ax.axhline(0.90, color="crimson", linestyle="--", label="Target 0.90")
ax.set_ylim(0.0, 1.05)
ax.set_title("Cakupan Empiris P90 Out-of-Time (Sebelum vs Sesudah Conformal)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Rincian Error per Segmen & Interpretabilitas SHAP / Partial Dependence (R1e)

**Tujuan:** Menganalisis error (`pinball_p50`, `pinball_p90`, `mae`, `rmse`, `p90_hit_rate`) berdasarkan `layanan`, `pulau_tujuan`, dan `mitra_kurir` (termasuk perilaku pada kurir baru `K13` di data uji), serta menjelaskan penggerak utama model `p50` menggunakan **SHAP (`TreeExplainer`)** dan **Partial Dependence Plot (PDP)**.

In [ ]:
full_prep = best_model_bundle["full_preprocessor"]
test_views = full_prep.transform(test_clean)
test_p50_raw, test_p90_raw = _predict_with_bundle(best_model_bundle["full_bundle"], test_views)
test_p50, test_p90 = apply_conformal_calibration(
    test_p50_raw,
    test_p90_raw,
    q_hat=q_hat,
    mode=str(calib_summary["selected_space"]),
)

seen_mitra = set(train_clean["mitra_kurir"].astype(str).unique())
breakdowns = compute_group_breakdowns(
    val_df_meta=train_clean.iloc[oot_val_idx],
    y_val=best_model_bundle["oot_y"],
    p50_val=best_model_bundle["oot_p50_raw"],
    p90_val=oot_p90_cal,
    test_df=test_clean,
    p50_test=test_p50,
    p90_test=test_p90,
    seen_mitra=seen_mitra,
)

print("--- Rincian Error OOT per Layanan ---")
display(pd.DataFrame(breakdowns["by_layanan"]))
print("--- Rincian Error OOT per Pulau Tujuan ---")
display(pd.DataFrame(breakdowns["by_pulau_tujuan"]))
print("--- Ringkasan Prediksi Uji per Mitra Kurir (termasuk K13 unseen) ---")
display(pd.DataFrame(breakdowns["test_mitra_kurir_summary"]))

interp_summary, saved_artifacts = compute_interpretability(
    best_model_bundle=best_model_bundle,
    seed=SEED,
    smoke_cfg=smoke_cfg,
    cv_summary=cv_summary,
    model_results=model_results,
    calib_summary=calib_summary,
    oot_p90_cal=oot_p90_cal,
    breakdowns=breakdowns,
    artifacts_dir=ARTIFACTS_DIR,
)

print("\nTop 10 Fitur Penggerak Utama (SHAP):")
display(pd.DataFrame(interp_summary["top_drivers"]).head(10))
print("Ringkasan Partial Dependence (dalam jam):")
display(pd.DataFrame(interp_summary["pdp_summary"]))
print("Artefak grafik tersimpan di:", ARTIFACTS_DIR)
for k, p in saved_artifacts.items():
    print(f"  - {k}: {p}")

## 7. Ekspor & Verifikasi `prediksi_regresi.csv` (R1f)

**Tujuan:** Menyimpan hasil prediksi akhir ke `prediksi_regresi.csv` dengan header persis `id_paket,p50,p90`, jumlah baris dan urutan ID persis sama dengan `regresi_test.csv.gz`, tanpa NaN/Inf, `p50 > 0`, dan `p90 >= p50`.

In [ ]:
written_csv = export_regresi_csv(
    ids=test_clean["id_paket"],
    p50=test_p50,
    p90=test_p90,
    output_csv=OUTPUT_CSV,
)

assert Path(written_csv).name == "prediksi_regresi.csv"
df_sub = pd.read_csv(written_csv, encoding="utf-8")
assert list(df_sub.columns) == ["id_paket", "p50", "p90"], f"Header salah: {list(df_sub.columns)}"
assert len(df_sub) == len(test_clean), f"Jumlah baris tidak cocok: {len(df_sub)} vs {len(test_clean)}"
assert (df_sub["id_paket"].astype(str).to_numpy() == test_clean["id_paket"].astype(str).to_numpy()).all()
assert np.isfinite(df_sub["p50"]).all() and np.isfinite(df_sub["p90"]).all()
assert (df_sub["p50"] > 0).all() and (df_sub["p90"] >= df_sub["p50"]).all()

print(f"File submission berhasil ditulis: {written_csv} ({len(df_sub):,} baris)")
display(df_sub.head(10))
display(df_sub[["p50", "p90"]].describe())

## 8. Ringkasan Metodologi, Keterbatasan, & Reproduksibilitas (R3)

- **Anti-Kebocoran & Audit**: Fitur pasca-pickup `jumlah_scan` dibuang penuh; kesalahan satuan `durasi_jam > 1000` diperbaiki (`/ 60.0`) hanya di data latih; sentinel `jarak_km` (`0` / `9999`) dan missingness MNAR `berat_kg` diimputasi ketat di dalam fold latih.
- **Validasi Temporal**: Seluruh tuning hyperparameter dan pemilihan model dilakukan di dalam *expanding-window next-month Time CV* (`train_max < val_min`) dan dibandingkan terhadap Random K-Fold CV.
- **Kategori Kardinalitas Tinggi & Baru**: `kota_tujuan` (405 level) di-encode dengan OOF `TargetEncoder` + frekuensi latih; kurir baru `K13` pada data uji jatuh aman ke prior global dan frekuensi `0.0`.
- **Kalibrasi Ketidakpastian**: Split Conformal Prediction pada fold OOT memastikan cakupan empiris P90 mendekati 90% dengan jaminan `p90 >= p50`.
- **Reproduksi Satu Perintah**: Jalankan `python kode/run_all.py` (atau `python -m kode.regresi` khusus Task 1) dari root proyek menggunakan `.venv`.